# Automated ML
## Introduction

This notebook is automatically generated by the Fabric low-code AutoML wizard based on your selections. Whether you're building a regression model, a classifier, or another machine-learning solution, this tool simplifies the process by transforming your goals into executable code. You can easily modify any settings or code snippets to better align with your requirements.

### What is FLAML?

[FLAML (Fast and Lightweight Automated Machine Learning)](https://aka.ms/fabric-automl) is an open-source AutoML library designed to quickly and efficiently find the best machine learning models and hyperparameters. FLAML optimizes for speed, accuracy, and cost, making it an excellent choice for a wide range of machine-learning tasks.

### Steps in this notebook

1. **Load the data**: Import your dataset.
2. **Generate features**: Automatically transform and preprocess your data to improve model performance.
3. **Use AutoML to find your best model**: Use FLAML to automatically select the most suitable model and optimize its parameters.
4. **Save the final machine learning model**: Store the trained model for future use.
5. **Generate predictions**: Use the saved model to predict outcomes on new data.

> [!IMPORTANT]
> **Automated ML is currently supported on Fabric Runtimes 1.2+ or any Fabric environment with Spark 3.4+.**

### Default notebook optimization

This cell configures the logging and warning settings to reduce unnecessary output and focus on critical information. It suppresses specific warnings and logs from the underlying libraries, ensuring a cleaner and more readable notebook experience.

In [ ]:
import logging
import warnings
 
logging.getLogger('synapse.ml').setLevel(logging.CRITICAL)
logging.getLogger('mlflow.utils').setLevel(logging.CRITICAL)
warnings.simplefilter('ignore', category=FutureWarning)
warnings.simplefilter('ignore', category=UserWarning)

## Step 1: Load the Data

This cell is responsible for importing the raw data from the specified source into the notebook environment. The data could come from various sources, such as a file or table in your lakehouse.

Once loaded, this data will serve as the input for subsequent steps, such as data transformation, model training, and evaluation.

In [ ]:
import re
df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(
    "abfss://aPandas@msit-onelake.dfs.fabric.microsoft.com/aPandas.Lakehouse/Files/churn.csv"
).cache()
df = df.toDF(*(re.sub('[^A-Za-z0-9_]+', '_', c) for c in df.columns))  # Replace not supported characters in column name with underscore to avoid invalid character for model training and saving

target_col = re.sub('[^A-Za-z0-9_]+', '_', "countryOrRegion")


In [ ]:
display(df)

## Step 2: Generate features

Featurization is the process of transforming raw data into a format optimized for training a machine learning model. It ensures the model can access the most relevant information, significantly impacting its accuracy and performance.

This step applies various techniques to refine the data, enhance its quality, and make it compatible with the selected algorithms, helping the model learn patterns more effectively.

In [ ]:
# Handle class imbalance
import matplotlib.pyplot as plt
from pyspark.sql import functions as F


distribution = (
    df.groupBy(target_col)
    .agg(F.count("*").alias("count"))
    .withColumn("proportion", F.col("count") / df.count())
)
distribution = distribution.toPandas()
distribution = distribution.set_index(target_col)['proportion']
dominant_class_proportion = distribution.max()

distribution.plot(kind='bar')
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Proportion")
plt.show()

if dominant_class_proportion > 0.8:
    print(f"The dataset is imbalanced. The dominant class has {dominant_class_proportion * 100:.2f}% of the samples.")
    print("You may need to handle class imbalance before training the model.")
    print("You can use techniques such as oversampling, undersampling, or using class weights to handle class imbalance.")
    print("For more information, see https://aka.ms/smote-example")
else:
    print("The dataset is balanced.")


In [ ]:
# Set Functions if needed for Featurization
from pyspark.sql.types import *

def filter_supported_columns(df, feature_cols):
    supported_types = (FloatType, DoubleType, ShortType, IntegerType, LongType)  # Types supported by VectorAssembler
    supported_cols = [col_name for col_name in feature_cols
                      if isinstance(df.schema[col_name].dataType, supported_types)]
   
    return supported_cols


In [ ]:
# Import the necessary library for feature vectorization
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import Imputer
from pyspark.ml import Pipeline
from pyspark.sql.functions import col
from flaml.automl.spark.utils import to_pandas_on_spark
from flaml.automl.data import auto_convert_dtypes_spark


cast_type = [['countryOrRegion', 'IntegerType'],['isPaidTimeOff', 'FloatType']]
for col, coltype in cast_type:
    col = re.sub('[^A-Za-z0-9_]+', '_', col)
    df = df.withColumn(col, df[col].cast(coltype()))

# convert string type to nearest dtype
df, _ = auto_convert_dtypes_spark(df)

# Identify columns with at least one non-null value
non_null_columns = [col_name for col_name in df.columns if df.filter(col(col_name).isNotNull()).count() > 0]
df = df.select(*non_null_columns)

# Train-Test Separation
train_raw, test_raw = df.randomSplit([0.8, 0.2], seed=41)

mean_features, median_features, mode_features = [], [], []

# Group columns by type
median_features = ['normalizeHolidayName']
median_features = [re.sub('[^A-Za-z0-9_]+', '_', col) for col in median_features]
 
all_features = mean_features + median_features + mode_features
mean_features = [
    col
    for col in filter_supported_columns(train_raw, train_raw.columns)
    if col not in all_features
] + mean_features

# Create a pipeline
pipeline = Pipeline(stages=[
    Imputer(strategy="median", inputCols=median_features, outputCols=median_features), 
])
# Fit and transform the data
model = pipeline.fit(train_raw)
train_raw = model.transform(train_raw)
test_raw = model.transform(test_raw)
# Fill NA values with 'null' for the string columns
train_raw = train_raw.fillna('null')
test_raw = test_raw.fillna('null')

# Fill NA values with False for the boolean columns
train_raw = train_raw.fillna(False)
test_raw = test_raw.fillna(False)

# Fill NA values with 0 for the numeric columns
train_raw = train_raw.fillna(0)
test_raw = test_raw.fillna(0)

# Define the feature columns (excluding the target_name variable "countryOrRegion")
feature_cols = [col for col in df.columns if col != target_col]
feature_cols = filter_supported_columns(df, feature_cols)

# Create a VectorAssembler to combine feature columns into a single 'features' column
featurizer = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="keep")

# Transform the training and testing datasets using the VectorAssembler
train_data = featurizer.transform(train_raw)[target_col, "features"]
test_data = featurizer.transform(test_raw)[target_col, "features"]

# Transform to pandas on spark according to the selected models
df_train = to_pandas_on_spark(train_data)
df_test = to_pandas_on_spark(test_data)

display(df_train[:10])


## Step 3: Use AutoML to find your best model

We will now use FLAML's AutoML to automatically find the best machine learning model for our data. AutoML (Automated Machine Learning) simplifies the model selection process by automatically testing and tuning various algorithms and configurations, helping us quickly identify the most effective model with minimal manual effort.

### Tracking results with experiments in Fabric

Experiments in Fabric let you track the results of your AutoML process, providing a comprehensive view of all the metrics and parameters from your trials.

In [ ]:
# MLFlow Logging Related

import mlflow

mlflow.autolog(exclusive=False)
mlflow.set_experiment("experiment-signature")


#### Configure the AutoML trial and settings

These configurations are driven by the AutoML mode and task selected in the wizard. For example, if you select "quick prototype", you'll see a setting for time budget.

In [ ]:
# Import the AutoML class from the FLAML package
import flaml
from flaml import AutoML

# Define AutoML settings
settings = {
    "time_budget": 2, # Total running time in seconds, -1 to unlimit 
    "metric": "r2", 
    "task": "binary", 
    "log_file_name": "flaml_experiment.log",  # FLAML log file
    "eval_method": "cv",
    "n_splits": 3,
    "max_iter": 10, 
    "early_stop": True, 
    "seed": 41 , # Random seed 
    "mlflow_exp_name": "experiment-signature",  # MLflow experiment name
    "verbose": 1, 
    "featurization": "auto", 
}

if flaml.__version__ > "2.3.3":
    settings["entrypoint"] = "low-code"

# Create an AutoML instance
automl = AutoML(**settings)


#### Run the AutoML trial

Run the AutoML trial, with all trials being tracked as experiment runs. The trial is performed on the processed dataset, using the `Exited` variable as the target, and applying the defined configurations for optimal model selection.

In [ ]:
with mlflow.start_run(nested=True, run_name="model_name_test"):
    automl.fit(
        dataframe=df_train, 
        label=target_col, 
    )

## Step 4: Save the final machine learning model

Upon completing the AutoML trial, you can now save the final, tuned model as an ML model in Fabric.

In [ ]:
model_path = f"runs:/{automl.best_run_id}/model"

# Register the model to the MLflow registry
registered_model = mlflow.register_model(model_uri=model_path, name="model_name_test")

# Print the registered model's name and version
print(f"Model '{registered_model.name}' version {registered_model.version} registered successfully.")

## Step 5: Generate predictions

Microsoft Fabric lets you operationalize machine learning models with a scalable function called `PREDICT`, which supports batch scoring (or batch inferencing) in any compute engine. You can generate batch predictions directly from the Microsoft Fabric notebook or from a given ML model's item page. For more information on how to use `PREDICT`, see [Model scoring with PREDICT in Microsoft Fabric](https://aka.ms/fabric-predict).

1. Generate predictions.

In [ ]:
model_name = "model_name_test"
model = mlflow.spark.load_model(f"models:/{registered_model.name}/{registered_model.version}")

df_test = df_test.to_spark()
batch_predictions = model.transform(df_test)


In [ ]:
display(batch_predictions)

2. Save the predictions to a table.

In [ ]:
saved_name = "Tables/publicholidays_predictions".replace(".", "_")
batch_predictions.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(saved_name)